In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [4]:
def map_benchmark_name(benchmark):
    """Map a benchmark name to the shorthand name in the paper."""
    if benchmark == "InvariantMassSequential":
        return "SIM"
    elif benchmark == "InvariantMassRandom":
        return "RIM"

    return benchmark

In [2]:
zen2 = pd.read_csv("/data/data-layout-benchmarks/Europar26/AMD_Zen_2.csv")
zen4 = pd.read_csv("/data/data-layout-benchmarks/Europar26/AMD_Zen_4.csv")
hsw = pd.read_csv("/data/data-layout-benchmarks/Europar26/Intel_Haswell.csv")
skl = pd.read_csv("/data/data-layout-benchmarks/Europar26/Intel_Skylake.csv")
dfs = [zen2, zen4, hsw, skl]
labels = ["AMD Zen 2", "AMD Zen 4", "Intel Haswell", "Intel Skylake"]

In [154]:
# TABLE 2
def format_sig(x, sig=4):
    if x == 0:
        return f"{0:.{sig-1}f}"
    decimals = max(sig - int(np.floor(np.log10(abs(x)))) - 1, 0)
    return f"{x:.{decimals}f}"

minmax_items = []
for di, (df, lbl) in enumerate(zip(dfs, labels)):
    for benchmark in df["benchmark"].unique():
        for pi, problem_size in enumerate(df["problem_size"].unique()):
            df_bp = df[
                (df["benchmark"] == benchmark) & (df["problem_size"] == problem_size)
            ]
            time_vals = df_bp.groupby("container")["time"].mean()
            min_container = df_bp.groupby("container")["time"].mean().idxmin()
            max_container = df_bp.groupby("container")["time"].mean().idxmax()
            min_string = (
                r"\multicolumn{1}{r}{\begin{tabular}[r]{@{}r@{}}"
                + format_sig(time_vals.min())
                + r"\\ \texttt{"
                + min_container.replace("PartitionedContainer", "").replace("_", "\\_")
                + r"}\end{tabular}}"
            )
            max_string = (
                r"\multicolumn{1}{r"
                + ("|" if di < len(dfs) - 1 else "")
                + r"}{\begin{tabular}[r]{@{}r@{}}"
                + format_sig(time_vals.max())
                + r"\\ \texttt{"
                + max_container.replace("PartitionedContainer", "").replace("_", "\\_")
                + r"}\end{tabular}}"
            )
            minmax_items.append(
                [
                    lbl,
                    map_benchmark_name(benchmark),
                    "Small" if pi == 0 else "Large",
                    min_string,
                    max_string,
                ]
            )

minmax_items = pd.DataFrame(
    minmax_items, columns=["System", "Kernel", "Arrays", "Best", "Worst"]
)
out = minmax_items.pivot(
    index=["Kernel", "Arrays"], columns="System", values=["Best", "Worst"]
)
out = out.sort_index(level=["Kernel", "Arrays"], ascending=[False, False])
out = out.swaplevel(0, 1, axis=1)

arch_order = ["AMD Zen 2", "AMD Zen 4", "Intel Haswell", "Intel Skylake"]
out = out.reindex(arch_order, axis=1, level=0)

# Don't write index name for System
out.columns.names = [None, None]

latex = out.to_latex(escape=False, index_names=False, column_format=r"@{}rrrrrrrrrr@{}")
latex = latex.replace(
    r" &  & Best", r" & \multicolumn{1}{l|}{Arrays} & \multicolumn{1}{r}{Best}"
)
latex = latex.replace(r"Worst", r"\multicolumn{1}{r|}{Worst}")
latex = latex.replace(r"\multicolumn{1}{r|}{Worst} \\", r"\multicolumn{1}{r}{Worst} \\")
latex = latex.replace(r"Small", r" \multicolumn{1}{r|}{Small}")
latex = latex.replace(r"& Large", r"\multicolumn{1}{r|}{} & \multicolumn{1}{r|}{Large}")
latex = latex.replace(
    r"\multirow[t]{2}{*}{SIM}",
    r"\multicolumn{1}{c|}{\multirow{2}{*}{\rotatebox{90}{{SIM}}}}",
)
latex = latex.replace(
    r"\multirow[t]{2}{*}{RIM}",
    r"\multicolumn{1}{c|}{\multirow{2}{*}{\rotatebox{90}{{RIM}}}}",
)

# Add midrule after first row
latex = latex.replace(r"{Intel Skylake} \\", r"{Intel Skylake} \\ \midrule", 1)

latex = latex.replace("\\cline{1-10}\n\\bottomrule", r"\bottomrule", 1)

with open("all_minmax.tex", "w") as f:
    f.write(latex)

In [178]:
# TABLE 3
aos_container = "PartitionedContainer0123456"
soa_container = "PartitionedContainer0_1_2_3_4_5_6"

aos_soa_items = []
for di, (df, lbl) in enumerate(zip(dfs, labels)):
    for benchmark in df["benchmark"].unique():
        for pi, problem_size in enumerate(df["problem_size"].unique()):
            df_bp = df[
                (df["benchmark"] == benchmark) & (df["problem_size"] == problem_size)
            ]
            aos_vals = df_bp[df_bp["container"] == aos_container].groupby("container")["time"].mean()
            soa_vals = df_bp[df_bp["container"] == soa_container].groupby("container")["time"].mean()

            aos_string = format_sig(aos_vals.iloc[0])
            soa_string = format_sig(soa_vals.iloc[0])
            aos_soa_items.append(
                [
                    lbl,
                    map_benchmark_name(benchmark),
                    "Small" if pi == 0 else "Large",
                    aos_string,
                    soa_string,
                ]
            )

aos_soa_df = pd.DataFrame(
    aos_soa_items, columns=["System", "Kernel", "Arrays", "AoS", "SoA"]
)
out = aos_soa_df.pivot(
    index=["Kernel", "Arrays"], columns="System", values=["AoS", "SoA"]
)
out = out.sort_index(level=["Kernel", "Arrays"], ascending=[False, False])
out = out.swaplevel(0, 1, axis=1)

arch_order = ["AMD Zen 2", "AMD Zen 4", "Intel Haswell", "Intel Skylake"]
out = out.reindex(arch_order, axis=1, level=0)

latex = out.to_latex(escape=False, index_names=False,
                     column_format=r"l|l|@{\hspace{0.1cm}}r@{\hspace{0.1cm}}r|@{\hspace{0.1cm}}r@{\hspace{0.25cm}}r|@{\hspace{0.1cm}}r@{\hspace{0.25cm}}r|@{\hspace{0.1cm}}r@{\hspace{0.1cm}}r")

latex = latex.replace(r"& System &", r"\multicolumn{1}{c}{} & \multicolumn{1}{c}{} &", 1)
latex = latex.replace(r"&  & AoS ", r"Kernel  & Arrays & AoS", 1)
latex = latex.replace(r"{Intel Skylake} \\", r"{Intel Skylake} \\ \midrule", 1)
latex = latex.replace("\\cline{1-10}\n\\bottomrule", r"\bottomrule", 1)

with open("all_aos_soa.tex", "w") as f:
    f.write(latex)
